In [4]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd
from shapely.geometry import mapping

# --- File Paths ---
hull_files = [
    "Concave_Hull_5_min_Traffic_Bengaluru.gpkg",
    "Concave_Hull_5_min_Free_Bengaluru.gpkg",
    "Concave_Hull_10_min_Traffic_Bengaluru.gpkg",
    "Concave_Hull_10_min_Free_Bengaluru.gpkg",
    "Concave_Hull_15_min_Traffic_Bengaluru.gpkg",
    "Concave_Hull_15_min_Free_Bengaluru.gpkg"
]

pop_raster_path = "/content/Bengaluru_GHSL_2026_New.tif"
city_boundary_path = "/content/BBMP_UTM_Cleaned.gpkg"

# --- Helper Function for Rasterio ---
def calculate_population(gdf, raster_path):
    """Extracts and sums population pixels within a GeoDataFrame polygon."""
    # PROPER FIX: Use shapely.geometry.mapping to format for rasterio
    geoms = [mapping(geom) for geom in gdf.geometry]

    with rasterio.open(raster_path) as src:
        # Mask the raster to the polygon boundary
        out_image, out_transform = mask(src, geoms, crop=True, nodata=src.nodata)

        # Filter out NoData values and sum the remaining pixels
        if src.nodata is not None:
            valid_data = out_image[out_image != src.nodata]
        else:
            valid_data = out_image[out_image >= 0]

        return np.nansum(valid_data)

# --- 1. Calculate Baselines (Total City Area & Population) ---
city_gdf = gpd.read_file(city_boundary_path)
city_gdf = city_gdf.dissolve()  # Dissolve all inner geometries

# Total Area (Project to EPSG:32643 - UTM Zone 43N for Bengaluru)
city_gdf_proj = city_gdf.to_crs(epsg=32643)
total_area_sqkm = city_gdf_proj.geometry.area.sum() / 10**6

# Total Population (Match CRS of raster)
with rasterio.open(pop_raster_path) as src:
    city_gdf_raster_crs = city_gdf.to_crs(src.crs)
total_population = calculate_population(city_gdf_raster_crs, pop_raster_path)

print(f"Baseline - Total Area: {total_area_sqkm:.2f} sq km | Total Pop: {total_population:,.0f}\n")

# --- 2. Process Each Concave Hull ---
results = []

for file in hull_files:
    # Load and dissolve hull to prevent double-counting overlapping areas
    hull_gdf = gpd.read_file(file)
    hull_gdf = hull_gdf.dissolve()

    # Calculate Area
    hull_proj = hull_gdf.to_crs(epsg=32643)
    hull_area_sqkm = hull_proj.geometry.area.sum() / 10**6
    area_coverage_pct = (hull_area_sqkm / total_area_sqkm) * 100

    # Calculate Population
    with rasterio.open(pop_raster_path) as src:
        hull_raster_crs = hull_gdf.to_crs(src.crs)
    hull_population = calculate_population(hull_raster_crs, pop_raster_path)
    pop_coverage_pct = (hull_population / total_population) * 100

    # Store results
    results.append({
        "Scenario": file.split(".")[0],
        "Area (sq km)": round(hull_area_sqkm, 2),
        "Area Coverage (%)": round(area_coverage_pct, 2),
        "Population Covered": int(hull_population),
        "Pop Coverage (%)": round(pop_coverage_pct, 2)
    })

# --- 3. Output Results ---
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

Baseline - Total Area: 709.23 sq km | Total Pop: 13,073,847

                             Scenario  Area (sq km)  Area Coverage (%)  Population Covered  Pop Coverage (%)
 Concave_Hull_5_min_Traffic_Bengaluru        214.06              30.18             6166728             47.17
    Concave_Hull_5_min_Free_Bengaluru        343.28              48.40             9096170             69.58
Concave_Hull_10_min_Traffic_Bengaluru        500.16              70.52            11905821             91.07
   Concave_Hull_10_min_Free_Bengaluru        565.77              79.77            12477141             95.44
Concave_Hull_15_min_Traffic_Bengaluru        575.98              81.21            12566242             96.12
   Concave_Hull_15_min_Free_Bengaluru        599.64              84.55            12717132             97.27


In [1]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd
from shapely.geometry import mapping

# --- File Paths ---
hull_files = [
    "Concave_Hull_5_min_Traffic_Pune.gpkg",
    "Concave_Hull_5_min_Free_Pune.gpkg",
    "Concave_Hull_10_min_Traffic_Pune.gpkg",
    "Concave_Hull_10_min_Free_Pune.gpkg",
    "Concave_Hull_15_min_Traffic_Pune.gpkg",
    "Concave_Hull_15_min_Free_Pune.gpkg"
]

pop_raster_path = "/content/Pune_Pimpri_GHSL.tif"
city_boundary_path = "/content/Unified_Pune_Pimpri_Fixed_UTM.gpkg"

# --- Helper Function for Rasterio ---
def calculate_population(gdf, raster_path):
    """Extracts and sums population pixels within a GeoDataFrame polygon."""
    # PROPER FIX: Use shapely.geometry.mapping to format for rasterio
    geoms = [mapping(geom) for geom in gdf.geometry]

    with rasterio.open(raster_path) as src:
        # Mask the raster to the polygon boundary
        out_image, out_transform = mask(src, geoms, crop=True, nodata=src.nodata)

        # Filter out NoData values and sum the remaining pixels
        if src.nodata is not None:
            valid_data = out_image[out_image != src.nodata]
        else:
            valid_data = out_image[out_image >= 0]

        return np.nansum(valid_data)

# --- 1. Calculate Baselines (Total City Area & Population) ---
city_gdf = gpd.read_file(city_boundary_path)
city_gdf = city_gdf.dissolve()  # Dissolve all inner geometries

# Total Area (Project to EPSG:32643 - UTM Zone 43N for Bengaluru)
city_gdf_proj = city_gdf.to_crs(epsg=32643)
total_area_sqkm = city_gdf_proj.geometry.area.sum() / 10**6

# Total Population (Match CRS of raster)
with rasterio.open(pop_raster_path) as src:
    city_gdf_raster_crs = city_gdf.to_crs(src.crs)
total_population = calculate_population(city_gdf_raster_crs, pop_raster_path)

print(f"Baseline - Total Area: {total_area_sqkm:.2f} sq km | Total Pop: {total_population:,.0f}\n")

# --- 2. Process Each Concave Hull ---
results = []

for file in hull_files:
    # Load and dissolve hull to prevent double-counting overlapping areas
    hull_gdf = gpd.read_file(file)
    hull_gdf = hull_gdf.dissolve()

    # Calculate Area
    hull_proj = hull_gdf.to_crs(epsg=32643)
    hull_area_sqkm = hull_proj.geometry.area.sum() / 10**6
    area_coverage_pct = (hull_area_sqkm / total_area_sqkm) * 100

    # Calculate Population
    with rasterio.open(pop_raster_path) as src:
        hull_raster_crs = hull_gdf.to_crs(src.crs)
    hull_population = calculate_population(hull_raster_crs, pop_raster_path)
    pop_coverage_pct = (hull_population / total_population) * 100

    # Store results
    results.append({
        "Scenario": file.split(".")[0],
        "Area (sq km)": round(hull_area_sqkm, 2),
        "Area Coverage (%)": round(area_coverage_pct, 2),
        "Population Covered": int(hull_population),
        "Pop Coverage (%)": round(pop_coverage_pct, 2)
    })

# --- 3. Output Results ---
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

Baseline - Total Area: 742.55 sq km | Total Pop: 6,498,444

                        Scenario  Area (sq km)  Area Coverage (%)  Population Covered  Pop Coverage (%)
 Concave_Hull_5_min_Traffic_Pune        112.26              15.12             2704593             41.62
    Concave_Hull_5_min_Free_Pune        181.76              24.48             3948415             60.76
Concave_Hull_10_min_Traffic_Pune        265.83              35.80             4840057             74.48
   Concave_Hull_10_min_Free_Pune        348.11              46.88             5737814             88.30
Concave_Hull_15_min_Traffic_Pune        357.08              48.09             5741760             88.36
   Concave_Hull_15_min_Free_Pune        426.13              57.39             6104191             93.93


In [3]:
import geopandas as gpd
import pandas as pd

# --- File Paths ---
files = [
    "Bengaluru_Joined_Road_Cleaned_27_August.gpkg",
    "Pune_Joined_Road_Cleaned_27_August.gpkg"
]

for file in files:
    print(f"\n{'='*50}\nProcessing: {file}\n{'='*50}")

    # 1. Load the dataset
    gdf = gpd.read_file(file)

    # 2. Project to UTM Zone 43N (EPSG:32643) for accurate meter calculations
    gdf = gdf.to_crs(epsg=32643)
    gdf['length_km'] = gdf.geometry.length / 1000

    # --- TASK 1: Road Composition (By Number of Segments) ---
    total_segments = len(gdf)

    # Group by road attribute ('highway') and count segments
    composition = gdf['highway'].value_counts().reset_index()
    composition.columns = ['highway', 'segment_count']
    composition['percentage'] = (composition['segment_count'] / total_segments) * 100

    print(f"Total Segments: {total_segments:,}")
    print("Road Composition (by number of segments):")
    print(composition.to_string(index=False, formatters={'percentage': '{:.2f}%'.format}))
    print("-" * 50)

    # --- TASK 2: Time and Delay Metrics ---
    # Prevent division by zero
    gdf['osm_speed'] = gdf['osm_speed'].replace(0, 0.001)
    gdf['dynamic_speed'] = gdf['dynamic_speed'].replace(0, 0.001)

    # Calculate base times
    gdf['normal_time_min'] = (gdf['length_km'] / gdf['osm_speed']) * 60
    gdf['traffic_time_min'] = (gdf['length_km'] / gdf['dynamic_speed']) * 60
    gdf['delay_min'] = gdf['traffic_time_min'] - gdf['normal_time_min']

    # 1. Total time lost in hrs
    total_time_lost_hrs = gdf['delay_min'].sum() / 60

    # 2. Average delay sec/km
    total_delay_sec = gdf['delay_min'].sum() * 60
    total_network_length_km = gdf['length_km'].sum()
    avg_delay_sec_per_km = total_delay_sec / total_network_length_km

    # 3. City-wide penalty in percentage
    total_normal_time_min = gdf['normal_time_min'].sum()
    total_traffic_time_min = gdf['traffic_time_min'].sum()
    city_wide_penalty_pct = ((total_traffic_time_min - total_normal_time_min) / total_normal_time_min) * 100

    print(f"Total time lost (Hours): {total_time_lost_hrs:,.2f} hrs")
    print(f"Average delay (Sec/Km):  {avg_delay_sec_per_km:,.2f} sec/km")
    print(f"City-wide penalty (%):   {city_wide_penalty_pct:,.2f}%")
    print("="*50)

    # Optional: Save back to gpkg if you want to keep the calculations
    # output_name = file.replace(".gpkg", "_with_Delays.gpkg")
    # gdf.to_crs(epsg=4326).to_file(output_name, driver="GPKG")


Processing: Bengaluru_Joined_Road_Cleaned_27_August.gpkg
Total Segments: 127,766
Road Composition (by number of segments):
       highway  segment_count percentage
   residential          92400     72.32%
       service          22976     17.98%
      tertiary           5218      4.08%
     secondary           2923      2.29%
       primary           2072      1.62%
         trunk            941      0.74%
  primary_link            342      0.27%
    trunk_link            276      0.22%
secondary_link            210      0.16%
      motorway            188      0.15%
 tertiary_link            139      0.11%
 motorway_link             81      0.06%
--------------------------------------------------
Total time lost (Hours): 107.99 hrs
Average delay (Sec/Km):  23.82 sec/km
City-wide penalty (%):   13.49%

Processing: Pune_Joined_Road_Cleaned_27_August.gpkg
Total Segments: 78,855
Road Composition (by number of segments):
       highway  segment_count percentage
   residential          423